# ST-OMR Meter V5-3J — Background Rescue Failure Forensics

TRAIN-only, read-only kök neden analizi. **LAUNCH yalnız bir kez.** Bağlantı koparsa yalnız STATUS kullan. Eğitim, threshold tuning, Historical Validation, First-30, V5 VAL ve FINAL_HOLDOUT erişimi yoktur.


## 1 — LAUNCH (yalnız bir kez)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys, time

RUNNER_HEAD = "3653d2d70aa186330542fc18e1fc0c9a9f01ca8f"
RUNNER_BLOB = "d33901331c5e9f5524164682ae13cdb4745ed24c"
FORENSICS_IMPLEMENTATION_HEAD = "c978b14fba23f91c60f06d2166bb23e87856d8d6"
V53I_HEAD = "88c7acc551fa2b00b1f877f6a839704d58825adb"
V53G_HEAD = "b36a9d2f5daade2c3568cac8cbc736ca75ca435f"
EXPECTED_V53I_REPORT_SHA256 = "448b807086bc9ee66d090fdf173ce54e3c5e2a133e60cf6ae0a791aed2717434"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
RUNNER_REL = "tools/meter_v5_3j_background_runner_v1.py"
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {
    "DATA_ROOT": DATA_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "M4A_ROOT": M4A_ROOT,
    "D10_ROOT": D10_ROOT,
}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE/PATH CHECK = PASS")

ANN = DATA_ROOT / "annotations"
SOURCE_REPORT = ANN / "v5_3g_authoritative_rescue_training_report.json"
SOURCE_ENVELOPE = ANN / f"v5_3g_execution_envelope_{V53G_HEAD}.json"
SOURCE_V53I_REPORT = ANN / "v5_3i_train_acceptance_gate_v1.json"
RESCUE_DIR = ANN / "v5_3g_authoritative_rescue_artifacts"
for name, path in {
    "V5-3G REPORT": SOURCE_REPORT,
    "V5-3H ENVELOPE": SOURCE_ENVELOPE,
    "V5-3I HOLD REPORT": SOURCE_V53I_REPORT,
}.items():
    if not path.is_file():
        raise RuntimeError(f"{name} bulunamadi: {path}")
if not RESCUE_DIR.is_dir():
    raise RuntimeError(f"RESCUE DIR bulunamadi: {RESCUE_DIR}")
import hashlib
if hashlib.sha256(SOURCE_V53I_REPORT.read_bytes()).hexdigest() != EXPECTED_V53I_REPORT_SHA256:
    raise RuntimeError("V5-3I HOLD report SHA mismatch")
print("SOURCE EVIDENCE CHECK = PASS")

CONTROL_DIR = ANN / "v5_3j_background_control"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
LOCK = CONTROL_DIR / f"launch_{FORENSICS_IMPLEMENTATION_HEAD}.json"
HEARTBEAT = CONTROL_DIR / f"heartbeat_{FORENSICS_IMPLEMENTATION_HEAD}.json"
PROGRESS = CONTROL_DIR / f"progress_{FORENSICS_IMPLEMENTATION_HEAD}.json"
LOG = CONTROL_DIR / f"background_{FORENSICS_IMPLEMENTATION_HEAD}.log"
RESULT = ANN / "v5_3j_rescue_failure_forensics_v1.json"

if RESULT.exists():
    raise RuntimeError(f"Existing V5-3J report blocks launch: {RESULT}")
if LOCK.exists():
    state = json.loads(LOCK.read_text(encoding="utf-8"))
    raise RuntimeError(
        "V5-3J launch lock already exists; second forensics process is forbidden. "
        f"status={state.get('status')} pid={state.get('pid')}"
    )
print("OUTPUT/LOCK GUARD = PASS")

SOURCE_REPO = Path("/content/st-omr-v5-3j-runner-source")
repo_url = f"https://github.com/{REPOSITORY}.git"
if SOURCE_REPO.exists():
    shutil.rmtree(SOURCE_REPO)
subprocess.check_call(["git", "clone", "--no-checkout", repo_url, str(SOURCE_REPO)])
subprocess.check_call(["git", "-C", str(SOURCE_REPO), "fetch", "origin", RUNNER_HEAD, "--depth", "1"])
fetched = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPO), "rev-parse", "FETCH_HEAD"],
    text=True,
).strip()
if fetched != RUNNER_HEAD:
    raise RuntimeError(f"runner FETCH_HEAD mismatch: {fetched}")
subprocess.check_call(["git", "-C", str(SOURCE_REPO), "checkout", "--detach", RUNNER_HEAD])
actual_head = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPO), "rev-parse", "HEAD"],
    text=True,
).strip()
if actual_head != RUNNER_HEAD:
    raise RuntimeError(f"runner HEAD mismatch: {actual_head}")
if subprocess.check_output(
    ["git", "-C", str(SOURCE_REPO), "status", "--porcelain"],
    text=True,
).strip():
    raise RuntimeError("runner source worktree dirty")
runner_path = SOURCE_REPO / RUNNER_REL
if not runner_path.is_file():
    raise RuntimeError(f"runner missing: {runner_path}")
actual_blob = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPO), "hash-object", RUNNER_REL],
    text=True,
).strip()
if actual_blob != RUNNER_BLOB:
    raise RuntimeError(f"runner blob mismatch: {actual_blob}")
runner_source = runner_path.read_text(encoding="utf-8")
compile(runner_source, str(runner_path), "exec")
if runner_source.count("forensics.run_rescue_failure_forensics_v1(") != 1:
    raise RuntimeError("runner forensics-call count changed")
for forbidden in (
    "run_authoritative_rescue_training_v1(",
    "execute_rescue_tensor_harness_v1(",
    "run_train_acceptance_gate_v1(",
    "run_historical_retention_gate(",
    "torch.optim.",
    ".backward(",
    "optimizer.step(",
):
    if forbidden in runner_source:
        raise RuntimeError(f"runner contains forbidden token: {forbidden}")
print("EXACT BACKGROUND RUNNER = PASS")
print("RUNNER HEAD =", RUNNER_HEAD)
print("RUNNER BLOB =", RUNNER_BLOB)

initial = {
    "schema": "st-omr-meter-v5-3j-background-launch-v1",
    "forensics_implementation_head": FORENSICS_IMPLEMENTATION_HEAD,
    "runner_head": RUNNER_HEAD,
    "runner_blob": RUNNER_BLOB,
    "status": "ALLOCATED",
    "allocated_at_utc": datetime.now(timezone.utc).isoformat(),
    "decision": None,
    "log_path": str(LOG),
    "heartbeat_path": str(HEARTBEAT),
    "progress_path": str(PROGRESS),
    "result_path": str(RESULT),
}
payload = (json.dumps(initial, indent=2, sort_keys=True) + "\n").encode("utf-8")
fd = os.open(str(LOCK), os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
try:
    os.write(fd, payload)
finally:
    os.close(fd)

log_handle = LOG.open("ab", buffering=0)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
try:
    proc = subprocess.Popen(
        [sys.executable, "-u", str(runner_path)],
        stdin=subprocess.DEVNULL,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        close_fds=True,
        env=env,
    )
finally:
    log_handle.close()

time.sleep(2)
if proc.poll() is not None:
    tail = LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-120:]
    raise RuntimeError(
        "V5-3J background runner exited immediately:\n" + "\n".join(tail)
    )
state = json.loads(LOCK.read_text(encoding="utf-8"))
print("V5-3J BACKGROUND LAUNCH = PASS")
print("PID =", state.get("pid", proc.pid))
print("STATUS =", state.get("status"))
print("LOG =", LOG)
print("HEARTBEAT =", HEARTBEAT)
print("PROGRESS =", PROGRESS)
print("RESULT =", RESULT)
print("Baglanti koparsa LAUNCH hucresini tekrar calistirma; yalniz STATUS hucresini kullan.")


## 2 — STATUS / İZLEME (salt okunur; istediğin kadar)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

FORENSICS_IMPLEMENTATION_HEAD = "c978b14fba23f91c60f06d2166bb23e87856d8d6"
MYDRIVE = Path("/content/drive/MyDrive")
if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

ANN = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN" / "annotations"
CONTROL_DIR = ANN / "v5_3j_background_control"
LOCK = CONTROL_DIR / f"launch_{FORENSICS_IMPLEMENTATION_HEAD}.json"
HEARTBEAT = CONTROL_DIR / f"heartbeat_{FORENSICS_IMPLEMENTATION_HEAD}.json"
PROGRESS = CONTROL_DIR / f"progress_{FORENSICS_IMPLEMENTATION_HEAD}.json"
LOG = CONTROL_DIR / f"background_{FORENSICS_IMPLEMENTATION_HEAD}.log"
RESULT = ANN / "v5_3j_rescue_failure_forensics_v1.json"

if not LOCK.is_file():
    raise RuntimeError("V5-3J launch receipt bulunamadi. LAUNCH hucresini yeniden calistirmadan once durumu incele.")
state = json.loads(LOCK.read_text(encoding="utf-8"))
print("=== V5-3J STATUS ===")
print("STATUS =", state.get("status"))
print("PID =", state.get("pid"))
print("DECISION =", state.get("decision"))
print("ALLOCATED =", state.get("allocated_at_utc"))
print("STARTED =", state.get("started_at_utc"))
print("COMPLETED =", state.get("completed_at_utc"))
print("ERROR =", state.get("error_type"), state.get("error_message"))
print("RESULT EXISTS =", RESULT.is_file())

if HEARTBEAT.is_file():
    heartbeat = json.loads(HEARTBEAT.read_text(encoding="utf-8"))
    print("HEARTBEAT =", heartbeat)
    stamp = heartbeat.get("utc")
    if stamp:
        try:
            hb = datetime.fromisoformat(stamp)
            age = (datetime.now(timezone.utc) - hb).total_seconds()
            print("HEARTBEAT AGE SECONDS =", round(age, 1))
        except Exception:
            pass
else:
    print("HEARTBEAT = NOT YET WRITTEN")

if PROGRESS.is_file():
    progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
    print("PROGRESS =", progress)
    done = progress.get("done")
    total = progress.get("total")
    if isinstance(done, int) and isinstance(total, int) and total > 0:
        print("PROGRESS PERCENT =", round(100.0 * done / total, 2))
else:
    print("PROGRESS = NOT YET WRITTEN")

if LOG.is_file():
    lines = LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n--- LOG TAIL (last 140 lines) ---")
    print("\n".join(lines[-140:]))


## 3 — FINAL FORENSICS RECEIPT (salt okunur; sonuç hazır olunca)


In [ ]:
from pathlib import Path
import hashlib, json

FORENSICS_IMPLEMENTATION_HEAD = "c978b14fba23f91c60f06d2166bb23e87856d8d6"
EXPECTED_V53I_REPORT_SHA256 = "448b807086bc9ee66d090fdf173ce54e3c5e2a133e60cf6ae0a791aed2717434"
MYDRIVE = Path("/content/drive/MyDrive")
if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

ANN = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN" / "annotations"
RESULT = ANN / "v5_3j_rescue_failure_forensics_v1.json"
LOCK = ANN / "v5_3j_background_control" / f"launch_{FORENSICS_IMPLEMENTATION_HEAD}.json"

if not RESULT.is_file():
    print("FORENSICS RECEIPT = NOT READY")
    if LOCK.is_file():
        state = json.loads(LOCK.read_text(encoding="utf-8"))
        print("BACKGROUND STATUS =", state.get("status"))
        print("ERROR =", state.get("error_type"), state.get("error_message"))
else:
    result = json.loads(RESULT.read_text(encoding="utf-8"))
    digest = hashlib.sha256(RESULT.read_bytes()).hexdigest()
    state = json.loads(LOCK.read_text(encoding="utf-8")) if LOCK.is_file() else {}
    if result.get("schema") != "st-omr-meter-v5-3j-rescue-failure-forensics-v1":
        raise RuntimeError("Unexpected V5-3J result schema")
    if result.get("bound_evidence", {}).get("v5_3i_report_sha256") != EXPECTED_V53I_REPORT_SHA256:
        raise RuntimeError("V5-3J result is not bound to the expected V5-3I HOLD report")

    print("FORENSICS RECEIPT = READY")
    print("BACKGROUND STATUS =", state.get("status"))
    print("V5-3I DECISION REPRODUCED =", result.get("v5_3i_decision_reproduced"))
    print("DIAGNOSIS SCOPE =", result.get("diagnosis_scope"))
    print()

    for digit in ("2", "3"):
        item = result["per_specialist"][digit]
        sig = item["failure_signature"]
        v5 = item["v5_train"]
        hist = item["historical_train"]
        v5_pos = v5["eligible_positive_rescue_probability"]
        hist_neg = hist["eligible_negative_rescue_probability"]
        print(f"=== {digit}-AI ===")
        print("FAILURE SIGNATURE =", sig["signature"])
        print("V5 POSITIVE RECOVERY FRACTION =", sig["v5_positive_recovery_fraction"])
        print("HISTORICAL TN REGRESSION COUNT =", sig["historical_true_negative_regression_count"])
        print("CROSS-DOMAIN V5_POS > HIST_NEG RANK FRACTION =", sig["cross_domain_v5_positive_over_historical_negative_rank_fraction"])
        print("CROSS-DOMAIN MEAN GAP =", sig["cross_domain_v5_positive_mean_minus_historical_negative_mean"])
        print("CROSS-DOMAIN MEDIAN GAP =", sig["cross_domain_v5_positive_median_minus_historical_negative_median"])
        print("SCORE ORDERING CONFLICT OBSERVED =", sig["score_ordering_conflict_observed"])
        print("FIXED THRESHOLD SEPARATES REQUIRED GROUPS =", sig["fixed_threshold_separates_required_groups"])
        print("V5 WITHIN-DOMAIN POS>NEG RANK =", v5["positive_over_negative_rank_fraction"])
        print("HIST WITHIN-DOMAIN POS>NEG RANK =", hist["positive_over_negative_rank_fraction"])
        print(
            "V5 POSITIVE SCORE DIST =",
            {k: v5_pos[k] for k in ("count", "min", "p05", "median", "p95", "max", "mean", "std_population")},
        )
        print(
            "HIST NEGATIVE SCORE DIST =",
            {k: hist_neg[k] for k in ("count", "min", "p05", "median", "p95", "max", "mean", "std_population")},
        )
        print("ACCEPTANCE WITNESS REPRODUCED =", item.get("v5_3i_acceptance_witness_reproduced"))
        print("GROUP IDENTITY REVERIFIED =", item.get("group_identity_reverified"))
        print()

    print("FROZEN STATE BIT IDENTICAL =", result.get("frozen_state_bit_identical"))
    print("RESCUE STATE BIT IDENTICAL DURING FORENSICS =", result.get("rescue_state_bit_identical_during_forensics"))
    print("REPAIR RECIPE SELECTED =", result.get("repair_recipe_selected"))
    print("RETRAINING AUTHORIZED =", result.get("retraining_authorized"))
    print("HISTORICAL VALIDATION OPENED =", result.get("historical_validation_opened"))
    print("FIRST-30 OPENED =", result.get("first30_opened"))
    print("V5 RESERVE OPENED =", result.get("v5_reserve_opened"))
    print("V5 VALIDATION OPENED =", result.get("v5_validation_opened"))
    print("FINAL_HOLDOUT LOCKED =", result.get("final_holdout_locked"))
    print("V5-3J REPORT SHA256 =", digest)
